# 11 · How much of each good roof is really free for panels? (Sedona geometry)

**Run this in Google Colab after notebook 10.** It takes about 20–40 minutes.

Until now every suitable roof counted with **60%** of its area (a guess). Here each roof's free area is
**calculated from its own outline**, for the 6.7 million roofs the model calls suitable:
1. **Edge strip:** the outline is shrunk inwards by 0.6 m (`ST_Buffer` with a negative distance), because panels
   are not placed at the very edge (walking, cleaning, safety, parapet wall).
2. **Stair room and water tank:** about 8 m² is taken off (a Kerala stair head-room is about 2.5 m × 3 m).
3. **Panels:** 10 m² of free roof per kW (MNRE rule of thumb, already includes the gaps between panel rows).

Small roofs lose a bigger share than big roofs, as in real life. Because 0.6 m and 8 m² are assumptions too,
it also shows a **sensitivity table** (edge 0.3 / 0.6 / 1.0 m × stair room 0 / 8 / 12 m²) next to the old 60% guess,
and a **realistic case** where every suitable roof installs at most 3 kW (the PM Surya Ghar subsidy limit).

Outputs: `data/features/usable_area` (per roof) and CSVs in `results/`. *(Module 1/2: Sedona spatial operations on 6.7 M polygons.)*

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the roof outlines and predictions to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
FEATURES = f'{PROJECT}/data/features'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'      # fast Colab disk, deleted when the session ends
for folder in (RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/11_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


t = time.time()
for name in ('roof_features', 'roof_predictions_gold'):
    shutil.copytree(f'{FEATURES}/{name}', f'{LOCAL}/{name}', dirs_exist_ok=True)
log(f'Roof outlines and predictions copied to {LOCAL} in {time.time() - t:.0f} s')

In [ ]:
# Step 2: install Java 17, PySpark 4.1, Apache Sedona 1.9.1 (about 2-5 minutes)
import subprocess


def java_version():
    java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java') if 'JAVA_HOME' in os.environ else 'java'
    out = subprocess.run(f'{java_bin} -version 2>&1', shell=True, capture_output=True, text=True).stdout
    return out.splitlines()[0] if out else 'no java'


if not any(f'version "{v}' in java_version() for v in ('17', '21')):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q pyspark==4.1.3 apache-sedona==1.9.1
log('Java:', java_version())

# Sedona's Java files, downloaded directly. Spark 4.1 ships its own copy of the JTS geometry library,
# so Sedona must sit on the same class path as Spark (see Step 3) instead of using spark.jars.packages.
import urllib.request

JARS = '/content/jars'
MAVEN = 'https://repo1.maven.org/maven2'
os.makedirs(JARS, exist_ok=True)
JAR_FILES = []
for url in (f'{MAVEN}/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar',
            f'{MAVEN}/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar'):
    path = f'{JARS}/{url.rsplit("/", 1)[1]}'
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    JAR_FILES.append(path)
log('Sedona JARs:', [os.path.basename(p) for p in JAR_FILES])

In [ ]:
# Step 3: start Spark with Sedona, using all CPU cores and most of the RAM
import psutil
from sedona.spark import SedonaContext

CORES = os.cpu_count()
MEM_GB = max(2, int(psutil.virtual_memory().total / 1e9 * 0.6))   # leave room for Python and the OS

config = (
    SedonaContext.builder()
    .master('local[*]')
    .appName('rooftop_usable_area')
    .config('spark.driver.memory', f'{MEM_GB}g')
    .config('spark.sql.shuffle.partitions', str(CORES * 4))
    .config('spark.driver.extraClassPath', ':'.join(JAR_FILES))
    .config('spark.sql.maxConcurrentOutputFileWriters', '20')   # write districts without sorting first
    .getOrCreate()
)
sedona = SedonaContext.create(config)
log(f'Spark {sedona.version} | CPU cores: {CORES} | Spark memory: {MEM_GB} GB')

In [ ]:
# Step 3b: open the Spark UI (click the link below while Step 4 runs, then screenshot "Jobs", "Stages" and "SQL")
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

In [ ]:
# Step 4: Sedona - shrink every suitable roof outline by the edge strip and measure what is left
SETBACKS = {'a03': 0.3, 'a06': 0.6, 'a10': 1.0}     # edge strip in metres (column name: area left after it)
OBSTACLES = [0, 8, 12]                               # stair room + water tank, m2
MAIN = ('a06', 8)                                    # main setting: 0.6 m edge, 8 m2 stair room and tank
M2_PER_KWP = 10
HOME_KW = 3                                          # realistic case: at most 3 kW per roof

roofs = sedona.read.format('geoparquet').load(f'{LOCAL}/roof_features').dropDuplicates(['id'])
roofs.createOrReplaceTempView('roofs')
sedona.read.parquet(f'{LOCAL}/roof_predictions_gold').filter('suitable').select('id', 'area_type') \
    .createOrReplaceTempView('suitable')

t = time.time()
buffers = ',\n           '.join(f'ST_Area(ST_Buffer(g, -{d})) AS {c}' for c, d in SETBACKS.items())
parts = sedona.sql(f"""
    WITH s AS (
        SELECT r.id, r.district, r.area_m2, r.pvout_kwh_per_kwp AS pvout, p.area_type,
               ST_Transform(r.geometry, 'EPSG:4326', 'EPSG:32643') AS g     -- UTM zone 43N, metres
        FROM roofs r JOIN suitable p ON r.id = p.id
    )
    SELECT id, district, area_m2, pvout, area_type,
           ST_Area(g) AS area_utm,
           {buffers}
    FROM s
""")
shutil.rmtree(f'{LOCAL}/usable_parts', ignore_errors=True)
parts.write.mode('overwrite').parquet(f'{LOCAL}/usable_parts')
parts = sedona.read.parquet(f'{LOCAL}/usable_parts').cache()
parts.createOrReplaceTempView('parts')
N = parts.count()
T_BUF = time.time() - t
log(f'Sedona: {N:,} suitable roofs transformed to metres and shrunk 3 ways in {T_BUF / 60:.1f} min '
    f'({N / T_BUF:,.0f} roofs per second)')

# Check: the area in metres must match the area from notebook 02 (if not, the coordinates were read the wrong way)
ratio = sedona.sql('SELECT percentile_approx(area_utm / area_m2, array(0.01, 0.5, 0.99)) AS q FROM parts '
                   'WHERE area_m2 > 0').collect()[0]['q']
log(f'CHECK area in metres / area from notebook 02 (1% / median / 99%): {[round(x, 4) for x in ratio]} (all near 1 is ideal)')
if not 0.98 < ratio[1] < 1.02:
    raise SystemExit('The metre areas do not match the earlier areas - stop and send this log to Claude.')

In [ ]:
# Step 5: results - free share of each roof, sensitivity table, potential per district
import pandas as pd


def usable(col, obst):
    return f'greatest({col} - {obst}, 0)'


MAIN_U = usable(*MAIN)

log('\nFREE SHARE OF SUITABLE ROOFS (main setting: 0.6 m edge, 8 m2 stair room and tank), by roof size')
log_table(sedona.sql(f"""
    SELECT CASE WHEN area_m2 < 50 THEN '1 small (<50 m2)' WHEN area_m2 < 120 THEN '2 medium (50-120 m2)'
                WHEN area_m2 < 500 THEN '3 large (120-500 m2)' ELSE '4 very large (>500 m2)' END AS roof_size,
           count(*) AS roofs,
           round(100 * sum({MAIN_U}) / sum(area_m2), 1) AS free_share_pct,
           transform(percentile_approx({MAIN_U} / area_m2, array(0.1, 0.5, 0.9)), x -> round(100 * x)) AS p10_p50_p90_pct,
           round(100 * avg(CAST({MAIN_U} = 0 AS DOUBLE)), 1) AS pct_with_no_room
    FROM parts GROUP BY 1
    UNION ALL
    SELECT '5 ALL', count(*), round(100 * sum({MAIN_U}) / sum(area_m2), 1),
           transform(percentile_approx({MAIN_U} / area_m2, array(0.1, 0.5, 0.9)), x -> round(100 * x)),
           round(100 * avg(CAST({MAIN_U} = 0 AS DOUBLE)), 1)
    FROM parts ORDER BY 1
""").toPandas(), 'usable_share_by_size.csv')

rows = [f"SELECT 'old guess: 60% of every suitable roof' AS method, round(100 * 0.6, 1) AS free_share_pct, "
        f"round(sum(area_m2 * 0.6) / {M2_PER_KWP} / 1e6, 2) AS gw, "
        f"round(sum(area_m2 * 0.6 / {M2_PER_KWP} * pvout) / 1e9, 2) AS twh_per_year FROM parts"]
for col, d in SETBACKS.items():
    for obst in OBSTACLES:
        u = usable(col, obst)
        tag = '  <- main' if (col, obst) == MAIN else ''
        rows.append(f"SELECT 'edge {d} m, stair room {obst} m2{tag}', round(100 * sum({u}) / sum(area_m2), 1), "
                    f"round(sum({u}) / {M2_PER_KWP} / 1e6, 2), round(sum({u} / {M2_PER_KWP} * pvout) / 1e9, 2) FROM parts")
log('\nSENSITIVITY: KERALA POTENTIAL OF THE SUITABLE ROOFS UNDER EACH ASSUMPTION')
log_table(sedona.sql(' UNION ALL '.join(rows)).toPandas(), 'usable_sensitivity.csv')

district = sedona.sql(f"""
    SELECT district, count(*) AS suitable_roofs,
           round(100 * sum({MAIN_U}) / sum(area_m2), 1) AS free_share_pct,
           round(sum({MAIN_U}) / {M2_PER_KWP} / 1e6, 2) AS gw,
           round(sum({MAIN_U} / {M2_PER_KWP} * pvout) / 1e9, 2) AS twh_per_year,
           round(sum(least({MAIN_U} / {M2_PER_KWP}, {HOME_KW})) / 1e6, 2) AS realistic_gw,
           round(sum(least({MAIN_U} / {M2_PER_KWP}, {HOME_KW}) * pvout) / 1e9, 2) AS realistic_twh_per_year
    FROM parts GROUP BY district ORDER BY twh_per_year DESC
""").toPandas()
total = {'district': 'KERALA', 'suitable_roofs': district.suitable_roofs.sum(),
         'free_share_pct': sedona.sql(f'SELECT round(100 * sum({MAIN_U}) / sum(area_m2), 1) FROM parts').collect()[0][0]}
for c in ('gw', 'twh_per_year', 'realistic_gw', 'realistic_twh_per_year'):
    total[c] = round(district[c].sum(), 2)
log('\nPOTENTIAL PER DISTRICT (main setting; realistic = at most 3 kW per suitable roof)')
log_table(pd.concat([district, pd.DataFrame([total])]), 'district_potential_usable.csv')

log('\nBY AREA TYPE (0 = homestead villages, 1 = towns, 2 = sparse open land)')
log_table(sedona.sql(f"""
    SELECT CAST(area_type AS INT) AS area_type, count(*) AS suitable_roofs,
           round(100 * sum({MAIN_U}) / sum(area_m2), 1) AS free_share_pct,
           round(sum({MAIN_U}) / {M2_PER_KWP} / 1e6, 2) AS gw,
           round(sum(least({MAIN_U} / {M2_PER_KWP}, {HOME_KW})) / 1e6, 2) AS realistic_gw
    FROM parts GROUP BY 1 ORDER BY 1
""").toPandas(), 'usable_by_area_type.csv')

In [ ]:
# Step 6: save the free area of every suitable roof (main setting) to Google Drive
t = time.time()
out = sedona.sql(f"""
    SELECT id, district, round({MAIN_U}, 1) AS usable_m2,
           round({MAIN_U} / {M2_PER_KWP}, 2) AS kwp_usable,
           round({MAIN_U} / {M2_PER_KWP} * pvout) AS kwh_year_usable
    FROM parts
""")
shutil.rmtree(f'{LOCAL}/usable_area', ignore_errors=True)
out.write.partitionBy('district').mode('overwrite').parquet(f'{LOCAL}/usable_area')
shutil.rmtree(f'{FEATURES}/usable_area', ignore_errors=True)
shutil.copytree(f'{LOCAL}/usable_area', f'{FEATURES}/usable_area')
log(f'\nSaved {FEATURES}/usable_area in {time.time() - t:.0f} s')
log(f'Sedona shrink step took {T_BUF / 60:.1f} min for {N:,} roofs')

# Data sizes on Google Drive, for the report's data-size table
import subprocess
sizes = subprocess.run(f'du -sh {PROJECT}/data/*/* | sort -h', shell=True, capture_output=True, text=True).stdout
log('\nDATA SIZES ON GOOGLE DRIVE\n' + sizes)
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/11_log.txt`.